# Travel Agent with a Remote MCP Server

This notebook builds a simple **travel agent** by connecting to a **remote MCP server** over HTTP. Unlike the local `stdio` server in `2.1_mcp.ipynb`, here we connect to a hosted server (Kiwi.com) using the `streamable_http` transport, load its flight-search tools, and let an agent use them to answer travel questions.

**Key learnings**
- Remote MCP over HTTP needs no local subprocess
- Checkpointers keep multi-turn travel planning coherent
- System prompts can constrain follow-up questions


### Load environment variables

`load_dotenv()` initializes the environment by loading the `.env` file, making API keys (such as the model provider key) available to the notebook.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

### Connect to the remote travel MCP server

We register a server named `travel_server` using the **`streamable_http` transport**, pointing at the hosted URL `https://mcp.kiwi.com`. Because it's an HTTP server, no subprocess is launched. `get_tools()` then loads the server's flight-search tools as LangChain tools.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

### Create the travel agent

`create_agent` builds a tool-calling agent using `gpt-5-nano` and the travel tools. We add an `InMemorySaver` **checkpointer** so conversation state is stored in memory across turns (keyed by `thread_id`), and a short `system_prompt` that sets the agent's role and tells it not to ask follow-up questions.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

### Ask the agent to find a flight

We send a `HumanMessage` requesting a direct flight from San Francisco to Tokyo. The `config` sets a `thread_id` so the checkpointer can track this conversation. The agent will call the remote MCP flight tools to search and return results. `ainvoke` runs the agent asynchronously.

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
    config
    )

### Inspect the full response

`pprint` pretty-prints the entire response, including the message history and any tool calls the agent made while searching for flights.

In [ ]:
from pprint import pprint

pprint(response)

### Show only the final answer

The response's `messages` list ends with the agent's final reply. `response["messages"][-1].content` prints just that last message, the human-readable flight recommendation.

In [ ]:
print(response["messages"][-1].content)